# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the Kilifi County Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and follows FAIR principles.

In [ ]:
# Install `mlcroissant` if not already installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Dataset Description: {dataset.metadata.description}")
print(f"Date Published: {dataset.metadata.datePublished}")
print(f"Keywords: {dataset.metadata.keywords}")

## 2. Data Overview
Review available record sets and fields. All references use their `@id` values for clarity and reproducibility.

The Croissant metadata describes survey data on mental health indicators, including demographic details and psychological assessment scores.

In [ ]:
# List available record sets by their @id
record_sets = dataset.record_sets
print("Available Record Sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', 'Unnamed record set')}")

# Explore fields within each record set
fields_dict = {}
for rs in record_sets:
    print(f"\nFields for Record Set {rs['@id']}:")
    if 'fields' in rs:
        fields_dict[rs['@id']] = [f['@id'] for f in rs['fields']]
        for f in rs['fields']:
            print(f" - {f['@id']} (name: {f.get('name', 'unnamed')}, type: {f.get('dataType', 'unknown')})")
    else:
        print("No fields defined.")

## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame for further analysis.
All references to data use the Croissant schema `@id` identifiers.

In [ ]:
# Choose a record set for extraction
# For demonstration, select the main survey record set. Replace with actual @id as discovered in previous step.
# If the only record set, use its @id (for example: 'cr:RecordSet1')

selected_record_set_id = record_sets[0]['@id'] if record_sets else None
if not selected_record_set_id:
    raise ValueError('No record sets available in the schema.')

# Load all records from the selected record set
records = list(dataset.records(record_set=selected_record_set_id))

# Convert to DataFrame
df = pd.DataFrame(records)

print(f"Loaded DataFrame Columns from Record Set {selected_record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group the data using specific fields and `@id` references.

We'll:
- Filter records on a numeric score (e.g. PHQ-9 score field `@id`)
- Normalize the score
- Group by a key demographic field (e.g. gender field `@id`)

**All fields referenced below are identified by their `@id`. If actual `@id`s are different, update accordingly.**

In [ ]:
# Example field @id values (replace with actual IDs from your overview)
phq9_score_id = None
gender_id = None
for col in df.columns:
    if 'phq9' in col.lower():
        phq9_score_id = col
    if 'gender' in col.lower():
        gender_id = col

if not phq9_score_id:
    # Fallback: choose the first numeric column
    phq9_score_id = df.select_dtypes(include=np.number).columns[0] if not df.empty else None
if not gender_id:
    gender_id = [col for col in df.columns if 'sex' in col.lower() or 'gender' in col.lower()][0] if df.columns.any() else None

print(f"Numeric Field (PHQ-9 Score) @id: {phq9_score_id}")
print(f"Grouping Field (Gender) @id: {gender_id}")

# Filter records where PHQ-9 score > threshold
threshold = 10
if phq9_score_id:
    filtered_df = df[df[phq9_score_id] > threshold]
    print(f"Filtered records with {phq9_score_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize PHQ-9 score
    filtered_df[f"{phq9_score_id}_normalized"] = (filtered_df[phq9_score_id] - filtered_df[phq9_score_id].mean()) / filtered_df[phq9_score_id].std()
    print(f"Normalized {phq9_score_id} for filtered records:")
    print(filtered_df[[phq9_score_id, f"{phq9_score_id}_normalized"]].head())

    # Group by gender (if present)
    if gender_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(gender_id)[phq9_score_id].mean().reset_index()
        print(f"Mean {phq9_score_id} grouped by {gender_id}:")
        print(grouped_df.head())
else:
    print("Could not find a suitable numeric score field for analysis.")

## 5. Visualization
Visualize the distribution of PHQ-9 scores and group means by gender.


In [ ]:
# Histogram for PHQ-9 score distribution
if phq9_score_id:
    plt.figure(figsize=(8, 4))
    df[phq9_score_id].dropna().hist(bins=20)
    plt.title(f"Distribution of {phq9_score_id} (PHQ-9 Score)")
    plt.xlabel("Score")
    plt.ylabel("Count")
    plt.show()

# Bar chart for mean PHQ-9 score per gender
if gender_id and phq9_score_id and gender_id in df.columns:
    group_means = df.groupby(gender_id)[phq9_score_id].mean().dropna()
    group_means.plot(kind='bar', figsize=(6,4))
    plt.title(f"Mean {phq9_score_id} by {gender_id}")
    plt.xlabel("Gender")
    plt.ylabel("Mean PHQ-9 Score")
    plt.show()

## 6. Conclusion
This notebook demonstrated loading, inspecting, and analyzing the Kilifi mental health survey dataset using the `mlcroissant` library. Referencing data using Croissant `@id` values ensures reproducibility and clarity.

- We loaded the dataset and examined available record sets and fields by `@id`.
- We extracted survey data, filtered by a psychological score, normalized scores, and grouped by gender.
- Visualizations revealed score distributions and demographic differences.

This approach offers a flexible, AI-ready workflow for working with FAIR datasets using Croissant standards.